In [1]:
import asyncio
import logging
import re

# Basics

Let's first take a look at what's inside the ``ib_insync`` package:

In [2]:
# to import local code
# https://stackoverflow.com/questions/61058798/python-relative-import-in-jupyter-notebook
# https://stackoverflow.com/questions/34478398/import-local-function-from-a-module-housed-in-another-directory-with-relative-im

import os, sys
notebook_dir = globals()['_dh'][0]  # Current notebook directory
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    # sys.path.append(parent_dir)
    sys.path.insert(0, parent_dir)
    print(parent_dir)


c:\Users\Jimmy\source\erdewit\ib_insync


In [3]:
import ib_insync
print(ib_insync.__version_info__)

(0, 9, 86)


In [4]:
import ib_insync.loop as loop_utils
import ib_insync.util as util

In [5]:
logging.getLogger('ib_insync.loop').setLevel(logging.DEBUG)
util.logToConsole(level=logging.DEBUG)

In [6]:
loop = loop_utils.getLoop(debug=True)

2025-10-26 20:31:27,705 ib_insync.loop INFO [getLoop] Loop source: running
2025-10-26 20:31:27,705 ib_insync.loop INFO [getLoop] Thread: MainThread
2025-10-26 20:31:27,706 ib_insync.loop INFO [getLoop] Loop type: ProactorEventLoop
2025-10-26 20:31:27,706 ib_insync.loop INFO [getLoop] Loop ID: 2526106719520
2025-10-26 20:31:27,707 ib_insync.loop INFO [getLoop] Loop is_running: True
2025-10-26 20:31:27,707 ib_insync.loop INFO [getLoop] Loop is_closed: False
2025-10-26 20:31:27,708 ib_insync.loop DEBUG [getLoop] Loop repr: <ProactorEventLoop running=True closed=False debug=False>


In [7]:
# ib_insync.ib.install_custom_repr_()

### Importing
The following two lines are used at the top of all notebooks. The first line imports everything and the second
starts an event loop to keep the notebook live updated:

In [8]:
from ib_insync import *
# util.startLoop()

*Note that startLoop() only works in notebooks, not in regular Python programs.*

### Connecting
The main player of the whole package is the "IB" class. Let's create an IB instance and connect to a running TWS/IBG application:

In [9]:
import socket

hostname = socket.gethostname()
local_ip = socket.gethostbyname(hostname)
print(f"Machine's IP Address: {local_ip}")

Machine's IP Address: 169.254.83.107


In [10]:
# util.logToConsole()
def apiStartEvent():
    print("API Start Event")

def apiEndEvent():
    print("API End Event")

def apiErrorEvent(errorMsg: str):
    print(f"API Error {errorMsg}")

def throttleStartEvent():
    print("Throttle Start Event")

def throttleEndEvent():
    print("Throttle End Event")

def onConnectedEvent():
    print("Connected Event")

def onDisconnectedEvent():
    print("Disconnected Event")

# def onNewOrderEvent(trade: Trade):
#     print(f"New Order Event {trade}")

# def onOpenOrderEvent(trade: Trade):
#     print(f"Open Order Event {trade}")

# def onOrderStatusEvent(trade: Trade):
#     print(f"Order Status Event {trade}")

# def onExecDetailsEvent(trade: Trade, fill: Fill):
#     print(f"Exec Details Event {trade} {fill}")

def onErrorEvent(reqId, errorCode, errorString, contract):
    print(f"Error Event {reqId} {errorCode} {errorString} {contract}")

def onTimeoutEvent(idlePeriod: float):
    print(f"Timeout Event {idlePeriod}")

# ib = IB()
# ib.client.apiStart.connect(apiStartEvent)
# ib.client.apiEnd.connect(apiEndEvent)
# ib.client.apiError.connect(apiErrorEvent)
# ib.client.throttleStart.connect(throttleStartEvent)
# ib.client.throttleEnd.connect(throttleEndEvent)


In [11]:
def on_accountDownloadEnd(account: str):
    print(f"Account Download End {account}")

def on_accountUpdateMultiEnd(reqId: int):
    print(f"Account Update Multi End {reqId}")

In [12]:
class LoggerFilter(logging.Filter):
    def __init__(self, logger_name, pattern=r'.*'):
        super().__init__()
        self.logger_name = logger_name
        self.pattern = re.compile(pattern)

    def filter(self, record: logging.LogRecord) -> bool:
        msg = record.getMessage()
        return not (record.name == self.logger_name and
            self.pattern.search(msg) and 
            record.levelno >= logging.INFO
        )


In [13]:
# # Set up logging
# logging.basicConfig(level=args.loglevel)
# logger = logging.getLogger(__name__)

wlogger = logging.getLogger('ib_insync.wrapper')
wlogger.setLevel(logging.DEBUG)
wlogger.addFilter(LoggerFilter('ib_insync.wrapper', 
    r'^(connectAck|nextValidId|execDetailsEnd|updateAccountTime|execDetails'
    r'|updateAccountValue|position|updatePortfolio|commissionReport|historicalData|Info 2104|Info 2106|Info 2158)'
))

In [14]:
# util.logToConsole(logging.DEBUG)

In [15]:
host_1 = '127.0.0.1'
host_2 = '192.168.1.90'
port_1 = 7497
port_2 = 4002
port_3 = 7496
# util.logToConsole(logging.DEBUG)
ib = ib_insync.IB()
ib.connectedEvent += onConnectedEvent
ib.disconnectedEvent += onDisconnectedEvent
ib.accountDownloadEndEvent += on_accountDownloadEnd
ib.accountUpdateMultiEndEvent += on_accountUpdateMultiEnd

ib.client.apiStart.connect(apiStartEvent)
ib.client.apiEnd.connect(apiEndEvent)
ib.client.apiError.connect(apiErrorEvent)
ib.client.throttleStart.connect(throttleStartEvent)
ib.client.throttleEnd.connect(throttleEndEvent)


2025-10-26 20:31:44,163 ib_insync.client DEBUG Client::__init__
2025-10-26 20:31:44,163 ib_insync.client DEBUG Client::reset


Event<throttleEnd, [[None, None, <function throttleEndEvent at 0x0000024C4757FCE0>]]>

In [ ]:

if not ib.isConnected():
    # ib.connect(host_1, port_3, clientId=102)
    asyncio.create_task(ib.connectAsync(host_1, port_3, clientId=102))
# ib.portfolio()
# ib.managedAccounts()

2025-10-26 20:32:12,259 ib_insync.client INFO Connecting to 127.0.0.1:7496 with clientId 102...
2025-10-26 20:32:12,262 ib_insync.client INFO Connected
2025-10-26 20:32:12,266 ib_insync.client DEBUG <<< 187,20251026 20:32:12 EST
2025-10-26 20:32:12,267 ib_insync.client DEBUG Client::isConnected
2025-10-26 20:32:12,268 ib_insync.client DEBUG >>> 71,2,102,
2025-10-26 20:32:12,269 ib_insync.client INFO Logged on to server version 187
2025-10-26 20:32:12,270 ib_insync.client DEBUG <<< 15,1,U2575725,U4372012,
2025-10-26 20:32:12,310 ib_insync.client DEBUG <<< 9,1,1
2025-10-26 20:32:12,311 ib_insync.client INFO API connection ready
2025-10-26 20:32:12,312 ib_insync.wrapper DEBUG startReq: key=positions
2025-10-26 20:32:12,312 ib_insync.client DEBUG Client::isConnected
2025-10-26 20:32:12,313 ib_insync.client DEBUG >>> 61,1
2025-10-26 20:32:12,314 ib_insync.wrapper DEBUG startReq: key=openOrders
2025-10-26 20:32:12,314 ib_insync.client DEBUG Client::isConnected
2025-10-26 20:32:12,315 ib_insy

API Start Event


2025-10-26 20:32:12,495 ib_insync.client DEBUG <<< 73,1,1,U2575725,,GrossPositionValue-S,2363.40,USD
2025-10-26 20:32:12,496 ib_insync.wrapper INFO accountUpdateMulti: AccountValue(account='U2575725', tag='GrossPositionValue-S', value='2363.40', currency='USD', modelCode='', lastUpdateTime=datetime.datetime(2025, 10, 26, 20, 32, 12, 496144, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=72000), 'Eastern Daylight Time'))), reqId 1
2025-10-26 20:32:12,496 ib_insync.client DEBUG <<< 73,1,1,U2575725,,Guarantee,0.00,USD
2025-10-26 20:32:12,497 ib_insync.wrapper INFO accountUpdateMulti: AccountValue(account='U2575725', tag='Guarantee', value='0.00', currency='USD', modelCode='', lastUpdateTime=datetime.datetime(2025, 10, 26, 20, 32, 12, 497144, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=72000), 'Eastern Daylight Time'))), reqId 1
2025-10-26 20:32:12,498 ib_insync.client DEBUG <<< 73,1,1,U2575725,,Guarantee-C,0.00,USD
2025-10-26 20:32:12,499 ib_insync.wrapper I

Account Update Multi End 1


2025-10-26 20:32:12,916 ib_insync.wrapper INFO accountUpdateMulti: AccountValue(account='U4372012', tag='PreviousDayEquityWithLoanValue', value='99262.01', currency='USD', modelCode='', lastUpdateTime=datetime.datetime(2025, 10, 26, 20, 32, 12, 916562, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=72000), 'Eastern Daylight Time'))), reqId 2
2025-10-26 20:32:12,916 ib_insync.client DEBUG <<< 73,1,2,U4372012,,PreviousDayEquityWithLoanValue-S,99262.01,USD
2025-10-26 20:32:12,917 ib_insync.wrapper INFO accountUpdateMulti: AccountValue(account='U4372012', tag='PreviousDayEquityWithLoanValue-S', value='99262.01', currency='USD', modelCode='', lastUpdateTime=datetime.datetime(2025, 10, 26, 20, 32, 12, 917563, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=72000), 'Eastern Daylight Time'))), reqId 2
2025-10-26 20:32:12,917 ib_insync.client DEBUG <<< 73,1,2,U4372012,,RegTEquity,99467.29,USD
2025-10-26 20:32:12,918 ib_insync.wrapper INFO accountUpdateMulti: AccountVa

Account Update Multi End 2
Connected Event


2025-10-26 20:32:23,396 ib_insync.client DEBUG <<< 73,1,1,U2575725,,Cushion,0.933539,
2025-10-26 20:32:23,397 ib_insync.wrapper INFO accountUpdateMulti: AccountValue(account='U2575725', tag='Cushion', value='0.933539', currency='', modelCode='', lastUpdateTime=datetime.datetime(2025, 10, 26, 20, 32, 23, 397173, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=72000), 'Eastern Daylight Time'))), reqId 1
2025-10-26 20:32:23,398 ib_insync.client DEBUG <<< 73,1,1,U2575725,,DayTradingStatus-S,20250224:20240223:true:660659.85::false,
2025-10-26 20:32:23,398 ib_insync.wrapper INFO accountUpdateMulti: AccountValue(account='U2575725', tag='DayTradingStatus-S', value='20250224:20240223:true:660659.85::false', currency='', modelCode='', lastUpdateTime=datetime.datetime(2025, 10, 26, 20, 32, 23, 398173, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=72000), 'Eastern Daylight Time'))), reqId 1
2025-10-26 20:32:23,399 ib_insync.client DEBUG <<< 73,1,1,U2575725,,SettledCashB

In [15]:
ib.TimezoneTWS

''

In [14]:
# ib.portfolio()

In [20]:
amd = Stock('AMD', 'SMART', 'USD')
uber = Stock('UBER', 'SMART', 'USD')
esfut = Future('ES', '202512', 'CME')
mesfut = Future('MES', '202512', 'CME')
mbtfut = Future('MBT', '202510', 'CME')

# cds = ib.reqContractDetails(amd)

# len(cds)
# cds[0]

In [19]:
Contract('FUT', symbol='ES', lastTradeDateOrContractMonth='202510', exchange='CME')

Contract(secType='FUT', symbol='ES', lastTradeDateOrContractMonth='202510', exchange='CME')

In [21]:
ib.qualifyContracts(mbtfut)[0]

2025-10-26 20:23:50,243 ib_insync.client DEBUG Client::getReqId 5
2025-10-26 20:23:50,243 ib_insync.wrapper DEBUG startReq: key=5 contract=Future(symbol='MBT', lastTradeDateOrContractMonth='202510', exchange='CME')
2025-10-26 20:23:50,244 ib_insync.client DEBUG Client::isConnected
2025-10-26 20:23:50,244 ib_insync.client DEBUG >>> 9,8,5,0,MBT,FUT,202510,0.0,,,CME,,,,,0,,,
2025-10-26 20:23:50,280 ib_insync.client DEBUG <<< 10,5,MBT,FUT,20251031 11:00:00 US/Central,20251031,0,,CME,USD,MBTV5,MBT,MBT,780341971,5,0.1,ACTIVETIM,AD,ADJUST,ALERT,ALGO,ALLOC,AVGCOST,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,ICE,IOC,LIT,LMT,LTH,MIT,MKT,MKTPROT,MTL,NGCOMB,NONALGO,OCA,PEGBENCH,SCALE,SCALERST,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,STPPROT,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF,CME,1,498814298,Micro Bitcoin,,202510,,,,US/Central,;20251026:1700-20251027:1600;20251027:1700-20251028:1600;20251028:1700-20251029:1600;20251029:1700-20251030:1600;20251030:1700-20251031:1100,

Future(conId=780341971, symbol='MBT', lastTradeDateOrContractMonth='20251031', lastTradeDate='20251031 11:00:00 US/Central', multiplier='0.1', exchange='CME', currency='USD', localSymbol='MBTV5', tradingClass='MBT')

In [22]:
mesfut

Future(symbol='MES', lastTradeDateOrContractMonth='202512', exchange='CME')

In [24]:
cdl = ib.reqContractDetails(mbtfut)

2025-10-26 20:24:03,066 ib_insync.client DEBUG Client::getReqId 6
2025-10-26 20:24:03,066 ib_insync.wrapper DEBUG startReq: key=6 contract=Future(conId=780341971, symbol='MBT', lastTradeDateOrContractMonth='20251031', lastTradeDate='20251031 11:00:00 US/Central', multiplier='0.1', exchange='CME', currency='USD', localSymbol='MBTV5', tradingClass='MBT')
2025-10-26 20:24:03,068 ib_insync.client DEBUG Client::isConnected
2025-10-26 20:24:03,068 ib_insync.client DEBUG >>> 9,8,6,780341971,MBT,FUT,20251031,0.0,,0.1,CME,,USD,MBTV5,MBT,0,,,
2025-10-26 20:24:03,071 ib_insync.client DEBUG <<< 10,6,MBT,FUT,20251031 11:00:00 US/Central,20251031,0,,CME,USD,MBTV5,MBT,MBT,780341971,5,0.1,ACTIVETIM,AD,ADJUST,ALERT,ALGO,ALLOC,AVGCOST,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,ICE,IOC,LIT,LMT,LTH,MIT,MKT,MKTPROT,MTL,NGCOMB,NONALGO,OCA,PEGBENCH,SCALE,SCALERST,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,STPPROT,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF,CME,1,498814298,Micro Bitcoin

In [25]:
cdl[0].tradingSessions()

[2025-10-26T17:00:00-05:00 - 2025-10-27T16:00:00-05:00,
 2025-10-27T17:00:00-05:00 - 2025-10-28T16:00:00-05:00,
 2025-10-28T17:00:00-05:00 - 2025-10-29T16:00:00-05:00,
 2025-10-29T17:00:00-05:00 - 2025-10-30T16:00:00-05:00,
 2025-10-30T17:00:00-05:00 - 2025-10-31T11:00:00-05:00]

In [21]:
cdl[0].liquidSessions()

[2025-01-13T08:30:00-06:00 - 2025-01-13T16:00:00-06:00,
 2025-01-14T08:30:00-06:00 - 2025-01-14T16:00:00-06:00,
 2025-01-15T08:30:00-06:00 - 2025-01-15T16:00:00-06:00,
 2025-01-16T08:30:00-06:00 - 2025-01-16T16:00:00-06:00,
 2025-01-17T08:30:00-06:00 - 2025-01-17T16:00:00-06:00]

In [ ]:
util.tree(cdl)

[{'ContractDetails': {'contract': {'Contract': {'secType': 'FUT',
     'conId': 780341971,
     'symbol': 'MBT',
     'lastTradeDateOrContractMonth': '20251031',
     'lastTradeDate': '20251031 11:00:00 US/Central',
     'multiplier': '0.1',
     'exchange': 'CME',
     'currency': 'USD',
     'localSymbol': 'MBTV5',
     'tradingClass': 'MBT'}},
   'marketName': 'MBT',
   'minTick': 5.0,
   'orderTypes': 'ACTIVETIM,AD,ADJUST,ALERT,ALGO,ALLOC,AVGCOST,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,ICE,IOC,LIT,LMT,LTH,MIT,MKT,MKTPROT,MTL,NGCOMB,NONALGO,OCA,PEGBENCH,SCALE,SCALERST,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,STPPROT,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF',
   'validExchanges': 'CME',
   'priceMagnifier': 1,
   'underConId': 498814298,
   'longName': 'Micro Bitcoin',
   'contractMonth': '202510',
   'timeZoneId': 'US/Central',
   'tradingHours': ';20251026:1700-20251027:1600;20251027:1700-20251028:1600;20251028:1700-20251029:1600;20251029:1700-2025103

2025-10-26 20:24:28,962 ib_insync.client DEBUG <<< 73,1,2,U4372012,,NetLiquidation,109051.13,USD
2025-10-26 20:24:28,963 ib_insync.wrapper INFO accountUpdateMulti: AccountValue(account='U4372012', tag='NetLiquidation', value='109051.13', currency='USD', modelCode='', lastUpdateTime=datetime.datetime(2025, 10, 26, 20, 24, 28, 963210, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=72000), 'Eastern Daylight Time'))), reqId 2
2025-10-26 20:24:28,963 ib_insync.client DEBUG <<< 8,1,20:24


In [23]:
# ib.disconnect()

In [24]:
# ib.positions()

In [25]:
# sp_: list[PortfolioItem] = [p for p in ib.portfolio() if p.contract.symbol in ['AAPL']]


In [26]:
# ib.portfolio()

In [27]:
ib.reqMarketDataType(1)

In [28]:
spy = Stock('SPY', 'ARCA', 'USD')


In [38]:
scvl = Stock('SCVL', 'SMART', 'USD')

In [39]:
ticker = ib.reqMktData(scvl, genericTickList='104') # 221,225,233,236,258,
for _ in range(20):
    if ticker.histVolatility and ticker.histVolatility > 0:
        break
    print("waiting for histVolatility")
    ib.sleep(.5)
ticker.histVolatility

waiting for histVolatility


0.48755153188756173

In [42]:
ticker = ib.reqMktData(scvl, genericTickList='59') # 221,225,233,236,258,
for _ in range(20):
    if ticker.dividends:
        print(ticker.dividends)
        break
    print("waiting for dividends")
    ib.sleep(.5)
ticker.dividends

waiting for dividends
Dividends(past12Months=0.54, next12Months=0.54, nextDate=datetime.date(2025, 4, 7), nextAmount=0.135)


Dividends(past12Months=0.54, next12Months=0.54, nextDate=datetime.date(2025, 4, 7), nextAmount=0.135)

In [41]:
ticker

Ticker(contract=Stock(symbol='SCVL', exchange='SMART', currency='USD'), time=datetime.datetime(2025, 1, 13, 18, 40, 26, 519111, tzinfo=datetime.timezone.utc), minTick=0.01, bid=30.19, bidSize=500.0, bidExchange='QN', ask=30.26, askSize=300.0, askExchange='QV', last=30.23, lastSize=100.0, lastExchange='D', prevBidSize=600.0, prevAskSize=200.0, volume=1592.0, open_=29.85, high=30.26, low=29.24, close=30.22, halted=0.0, histVolatility=0.48755153188756173, bboExchange='9c0001', snapshotPermissions=3)

In [31]:
ticker.histVolatility

nan

In [43]:
util.tree(ticker)

{'Ticker': {'contract': {'Stock': {'secType': 'STK',
    'symbol': 'SCVL',
    'exchange': 'SMART',
    'currency': 'USD'}},
  'time': '2025-01-13T13:40:54.791733-05:00',
  'minTick': 0.01,
  'bid': 30.19,
  'bidSize': 400.0,
  'bidExchange': 'Q',
  'ask': 30.26,
  'askSize': 300.0,
  'askExchange': 'Q',
  'last': 30.22,
  'lastSize': 100.0,
  'lastExchange': 'D',
  'prevBidSize': 500.0,
  'prevAskSize': 200.0,
  'prevLast': 30.19,
  'volume': 1600.0,
  'open_': 29.85,
  'high': 30.26,
  'low': 29.24,
  'close': 30.22,
  'halted': 0.0,
  'histVolatility': 0.4875581624424642,
  'dividends': {'past12Months': 0.54,
   'next12Months': 0.54,
   'nextDate': '2025-04-07',
   'nextAmount': 0.135},
  'bboExchange': '9c0001',
  'snapshotPermissions': 3}}

In [33]:
spy_ticker = ib.reqMktData(spy)

In [34]:
spy_ticker

Ticker(contract=Stock(symbol='SPY', exchange='ARCA', currency='USD'), time=datetime.datetime(2025, 1, 13, 18, 33, 25, 405408, tzinfo=datetime.timezone.utc), minTick=0.01, bid=579.76, bidSize=100.0, ask=579.78, askSize=100.0, last=579.76, lastSize=100.0, prevBid=579.73, prevBidSize=200.0, prevAsk=579.79, prevAskSize=200.0, prevLast=579.77, volume=232919.0, open_=575.75, high=580.08, low=575.35, close=580.49, halted=0.0, ticks=[TickData(time=datetime.datetime(2025, 1, 13, 18, 33, 25, 405408, tzinfo=datetime.timezone.utc), tickType=3, price=579.78, size=100.0)], bboExchange='a60001', snapshotPermissions=3)

In [35]:
util.tree(spy_ticker)

{'Ticker': {'contract': {'Stock': {'secType': 'STK',
    'symbol': 'SPY',
    'exchange': 'ARCA',
    'currency': 'USD'}},
  'time': '2025-01-13T13:33:30.227603-05:00',
  'minTick': 0.01,
  'bid': 579.74,
  'bidSize': 900.0,
  'ask': 579.77,
  'askSize': 200.0,
  'last': 579.76,
  'lastSize': 100.0,
  'prevBid': 579.76,
  'prevBidSize': 300.0,
  'prevAsk': 579.78,
  'prevAskSize': 300.0,
  'prevLast': 579.77,
  'volume': 232920.0,
  'open_': 575.75,
  'high': 580.08,
  'low': 575.35,
  'close': 580.49,
  'halted': 0.0,
  'bboExchange': 'a60001',
  'snapshotPermissions': 3}}

In [ ]:
if sp_: logger.info(f"Portfolio: {sp_[0].position} {sp_.contract.symbol}, avgCost {sp_[0].avgCost:,.0f}, marketPrice {sp_[0].marketPrice:.2f} marketValue {sp_[0].marketValue:,.0f}, unrealizedPNL {sp_[0].unrealizedPNL:,.0f} realizedPNL {sp_[0].realizedPNL:,.0f}")


In [15]:
ib.disconnect()

If the connection failed, then verify that the application has the API port enabled and double-check the hostname and port. For IB Gateway the default port is 4002. Make sure the clientId is not already in use.

If the connection succeeded, then ib will be synchronized with TWS/IBG. The "current state" is now available via methods such as ib.positions(), ib.trades(), ib.openTrades(), ib.accountValues() or ib.tickers(). Let's list the current positions:

In [9]:
# def accountValueEvent(accountValue: AccountValue):
#     print(accountValue)

In [10]:
# # Initialize an empty list to collect the data
# data_list = []

# # Define a function to handle the account value event
# async def collect_account_values():
#     global data_list
#     data_list = []
#     burst_active = False
#     burst_timeout = 3 * 60  # 3 minutes
#     burst_interval = 0.5  # 0.5 seconds

#     while True:
#         x = await asyncio.wait_for(account_value_queue.get(), timeout=burst_timeout)
#         if not burst_active:
#             burst_active = True
#             data_list = []
        
#         data_list.append(x)
        
#         # Wait for the next value or timeout
#         try:
#             x = await asyncio.wait_for(account_value_queue.get(), timeout=burst_interval)
#         except asyncio.TimeoutError:
#             burst_active = False
#             print("Burst ended. Collected data:", data_list)

# # Create a queue to collect account value events
# account_value_queue = asyncio.Queue()

# # Modify the accountValueEvent function to put the data into the queue
# def accountValueEvent(accountValue: AccountValue):
#     asyncio.create_task(account_value_queue.put(accountValue))
#     print(accountValue)

# # Start the collection process
# t = asyncio.create_task(collect_account_values())
# # loop = asyncio.get_event_loop()
# # loop.run_until_complete(t)

In [11]:
# ib.accountValueEvent += accountValueEvent

In [12]:
# ev = ib.accountValueEvent

In [13]:
# ev.takeuntil

In [ ]:
ib.accountSummary()

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

In [ ]:
ib.trades()

In [ ]:
trades = [t for t in ib.trades() if t.contract.symbol == 'AAPL']


In [ ]:
# ensure each trade has fills
tt = []
for trade in trades:
    if not trade.fills and trade.orderStatus.status != 'Cancelled':
        logger.error(f"Trade has no fills and orderStatus is not cancelled: {trade}")
    elif not trade.fills and trade.orderStatus.status == 'Cancelled':
        pass # ignore cancelled trades
        logger.info(f"Skipping cancelled trade: {trade}")
    else: # trade has fills
        # logger.debug(f"Trade has fills: {trade}")
        tt.append(trade)
# trades_df = ibtrades_to_df(t)
# logger.info(f"trades: {trades_df}")


In [ ]:
if tt:
    t = sorted(tt, key=lambda trade: max([fill.execution.time for fill in trade.fills]))
else:
    t = tt
if t: # sorted or empty
    for trade in reversed(t):
        if trade.order.action == 'SELL':
            lastSellTrade = trade
            lastSaleTime = max(fill.execution.time for fill in trade.fills)
            lastSalePrice = sum(fill.execution.price * fill.execution.shares for fill in trade.fills) / sum(fill.execution.shares for fill in trade.fills)
            logger.info(f"last sell trade: {trade}, last sale time: {lastSaleTime.astimezone()}, last sale price: {lastSalePrice}")
            break
    for trade in reversed(t):
        if trade.order.action == 'BUY':
            lastBuyTrade = trade
            lastBuyTime = max(fill.execution.time for fill in trade.fills)
            lastBuyPrice = sum(fill.execution.price * fill.execution.shares for fill in trade.fills) / sum(fill.execution.shares for fill in trade.fills)
            logger.info(f"last buy trade: {trade}, last buy time: {lastBuyTime.astimezone()}, last buy price: {lastBuyPrice}")
            break
else:
    logger.info(f"no trades found")


In [ ]:
def average_price(trade):
    return (sum(fill.execution.price * fill.execution.shares for fill in trade.fills) / sum(fill.execution.shares for fill in trade.fills)
            , max(fill.execution.time for fill in trade.fills))
def max_exec_time(trade):
    return max(fill.execution.time for fill in trade.fills)
def trade_commision(trade):
    return sum(fill.commissionReport.commission for fill in trade.fills)
def trade_realized_pnl(trade):
    return sum(fill.commissionReport.realizedPNL for fill in trade.fills)

In [ ]:
for trade in t:
    print(trade.order.action, trade.order.orderType, trade.order.lmtPrice, average_price(trade)[0], max_exec_time(trade).astimezone(), trade_commision(trade), trade_realized_pnl(trade))

In [ ]:
trades = ib.trades()

In [ ]:
trades[0]

In [ ]:
x = util.tree(trades)

In [ ]:
import json

In [ ]:
j = json.dumps(x)

In [ ]:
json.loads(j)

In [ ]:
ex = ib.executions()

In [ ]:
ex

In [ ]:
ex_df = util.df(ex)

In [ ]:
ex_df

In [ ]:
ulogger = logging.getLogger('ib_insync.util')


In [ ]:
ulogger.setLevel(logging.INFO)
util.tree(ex)

In [ ]:
ex[0].time.astimezone()

In [ ]:
# Function to flatten the trade structure
def flatten_trade(trade):
    trade_dict = {
        'contract_symbol': trade.contract.symbol,
        'contract_secType': trade.contract.secType,
        'contract_exchange': trade.contract.exchange,
        'contract_currency': trade.contract.currency,
        'order_action': trade.order.action,
        'order_totalQuantity': trade.order.totalQuantity,
        'order_orderType': trade.order.orderType,
        'order_lmtPrice': trade.order.lmtPrice,
        'order_auxPrice': trade.order.auxPrice,
        'order_tif': trade.order.tif,
        'order_status': trade.orderStatus.status,
        'order_filled': trade.orderStatus.filled,
        'order_remaining': trade.orderStatus.remaining,
        'order_avgFillPrice': trade.orderStatus.avgFillPrice,
        'order_lastFillPrice': trade.orderStatus.lastFillPrice,
        'order_permId': trade.order.permId,
        'order_clientId': trade.order.clientId,
        'order_orderId': trade.order.orderId,
        'order_parentId': trade.order.parentId,
        'order_whyHeld': trade.orderStatus.whyHeld,
        'order_mktCapPrice': trade.orderStatus.mktCapPrice
    }
    return trade_dict

# Flatten all trades
flattened_trades = [flatten_trade(trade) for trade in t]

# Create a hierarchical pandas DataFrame
df_trades = pd.DataFrame(flattened_trades)

print(df_trades)

In [ ]:
# Function to flatten the trade structure
def flatten_trade(trade):
    trade_dict = {
        'contract_symbol': trade.contract.symbol,
        'contract_secType': trade.contract.secType,
        'contract_exchange': trade.contract.exchange,
        'contract_currency': trade.contract.currency,
        'order_action': trade.order.action,
        'order_totalQuantity': trade.order.totalQuantity,
        'order_orderType': trade.order.orderType,
        'order_lmtPrice': trade.order.lmtPrice,
        'order_auxPrice': trade.order.auxPrice,
        'order_tif': trade.order.tif,
        'order_status': trade.orderStatus.status,
        'order_filled': trade.orderStatus.filled,
        'order_remaining': trade.orderStatus.remaining,
        'order_avgFillPrice': trade.orderStatus.avgFillPrice,
        'order_lastFillPrice': trade.orderStatus.lastFillPrice,
        'order_permId': trade.order.permId,
        'order_clientId': trade.order.clientId,
        'order_orderId': trade.order.orderId,
        'order_parentId': trade.order.parentId,
        'order_whyHeld': trade.orderStatus.whyHeld,
        'order_mktCapPrice': trade.orderStatus.mktCapPrice
    }
    return trade_dict

# Function to flatten the fill structure
def flatten_fill(fill):
    fill_dict = {
        'exec_execId': fill.execution.execId,
        'exec_time': fill.execution.time,
        'exec_acctNumber': fill.execution.acctNumber,
        'exec_exchange': fill.execution.exchange,
        'exec_side': fill.execution.side,
        'exec_shares': fill.execution.shares,
        'exec_price': fill.execution.price,
        'exec_permId': fill.execution.permId,
        'exec_clientId': fill.execution.clientId,
        'exec_orderId': fill.execution.orderId,
        'exec_liquidation': fill.execution.liquidation,
        'exec_cumQty': fill.execution.cumQty,
        'exec_avgPrice': fill.execution.avgPrice,
        'exec_orderRef': fill.execution.orderRef,
        'exec_evRule': fill.execution.evRule,
        'exec_evMultiplier': fill.execution.evMultiplier,
        'commission_report_commission': fill.commissionReport.commission,
        'commission_report_currency': fill.commissionReport.currency,
        'commission_report_realizedPNL': fill.commissionReport.realizedPNL,
        'commission_report_yield': fill.commissionReport.yield_,
        'commission_report_yieldRedemptionDate': fill.commissionReport.yieldRedemptionDate
    }
    return fill_dict

# Flatten all trades and fills
flattened_trades = [flatten_trade(trade) for trade in t]
flattened_fills = [flatten_fill(fill) for trade in t for fill in trade.fills]

# Create hierarchical pandas DataFrames
df_trades = pd.DataFrame(flattened_trades)
df_fills = pd.DataFrame(flattened_fills)

# Merge trades and fills DataFrames
df_merged = pd.merge(df_trades, df_fills, left_on='order_orderId', right_on='exec_orderId', how='outer')

print(df_merged)

In [ ]:
t[0]

In [ ]:
util.df(t)

In [ ]:
ib.reqCompletedOrders(False)

In [ ]:
ib.portfolio()

In [ ]:
ib.positions()

In [ ]:
ib.trades()

In [ ]:
t = [t for t in ib.trades() if t.contract.symbol == 'TLT']
t

In [ ]:
t = sorted([t for t in ib.trades() if t.contract.symbol == 'TLT'], key=lambda trade: max(fill.execution.time for fill in trade.fills))
t

In [ ]:
t1 = t[1]

In [ ]:
t1.fills

In [ ]:
t[0].contract.right

In [ ]:
import pprint
import json

In [ ]:
t[1].fills

In [ ]:
ib.openOrders()

In [ ]:
ib.reqAllOpenOrders()

Or filter the account values to get the liquidation value:

In [ ]:
v=ib.accountValues()[0]

In [ ]:
[v for v in ib.accountValues() if v.tag == 'NetLiquidationByCurrency' and v.currency == 'BASE']

The "current state" will automatically be kept in sync with TWS/IBG. So an order fill will be added as soon as it is reported, or account values will be updated as soon as they change in TWS.

### Contracts

Contracts can be specified in different ways:
* The ibapi way, by creating an empty Contract object and setting its attributes one by one;
* By using Contract and giving the attributes as keyword argument;
* By using the specialized Stock, Option, Future, Forex, Index, CFD, Commodity,
  Bond, FuturesOption, MutualFund or Warrant contracts.

Some examples:

In [31]:
c1 = Contract(conId=270639)
s1 = Stock('AMD', 'SMART', 'USD')
s2 = Stock('INTC', 'SMART', 'USD', primaryExchange='NASDAQ')
f1 = Forex('EURUSD')
c2 = CFD('IBUS30')
f2 = Future('ES', '20180921', 'GLOBEX')
f3 = Future('ES', '20241220', 'CME')
f4 = Future('ES', '202412', 'CME')
o1 = Option('SPY', '20241220', 580, 'C', 'SMART')
b1 = Bond(secIdType='ISIN', secId='US03076KAA60');
c3 = Commodity('XAUUSD', 'SMART', 'USD')

In [32]:
f5 = Forex(symbol='EUR', exchange='IDEALPRO', currency='USD')

In [14]:
mbt = Future('MBT', '202501', 'CME')

In [28]:
ib.qualifyContracts(mbt)

[Future(conId=718611434, symbol='MBT', lastTradeDateOrContractMonth='20250131', lastTradeDate='20250131 10:00:00 US/Central', multiplier='0.1', exchange='CME', currency='USD', localSymbol='MBTF5', tradingClass='MBT')]

Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.


Account Update Multi End 1
Account Download End DU9749485


Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.


Account Update Multi End 1
Account Download End DU9749485


Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: usfarm; secdefil. The following farms are not connected: ushmds.


Account Update Multi End 1


Peer closed connection.


In [26]:
s1d = ib.reqContractDetails(mbt)

In [16]:
util.tree(s1d)

[{'ContractDetails': {'contract': {'Contract': {'secType': 'FUT',
     'conId': 718611434,
     'symbol': 'MBT',
     'lastTradeDateOrContractMonth': '20250131',
     'lastTradeDate': '20250131 10:00:00 US/Central',
     'multiplier': '0.1',
     'exchange': 'CME',
     'currency': 'USD',
     'localSymbol': 'MBTF5',
     'tradingClass': 'MBT'}},
   'marketName': 'MBT',
   'minTick': 5.0,
   'orderTypes': 'ACTIVETIM,AD,ADJUST,ALERT,ALGO,ALLOC,AVGCOST,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,ICE,IOC,LIT,LMT,LTH,MIT,MKT,MTL,NGCOMB,NONALGO,OCA,PEGBENCH,SCALE,SCALERST,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF',
   'validExchanges': 'CME',
   'priceMagnifier': 1,
   'underConId': 498814298,
   'longName': 'Micro Bitcoin',
   'contractMonth': '202501',
   'timeZoneId': 'US/Central',
   'tradingHours': '20250111:CLOSED;20250112:1700-20250113:1600;20250113:1700-20250114:1600;20250114:1700-20250115:1600;20250115:1700-20250116

In [ ]:
f"{s1d[0].liquidSessions()[0].start} to {s1d[0].liquidSessions()[0].end}"

In [22]:
cdl = s1d[0].liquidSessions()

In [23]:
cdl[0].start

datetime.datetime(2025, 1, 13, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='US/Central'))

In [24]:
util.tree(s1d)

[{'ContractDetails': {'contract': {'Contract': {'secType': 'FUT',
     'conId': 718611434,
     'symbol': 'MBT',
     'lastTradeDateOrContractMonth': '20250131',
     'lastTradeDate': '20250131 10:00:00 US/Central',
     'multiplier': '0.1',
     'exchange': 'CME',
     'currency': 'USD',
     'localSymbol': 'MBTF5',
     'tradingClass': 'MBT'}},
   'marketName': 'MBT',
   'minTick': 5.0,
   'orderTypes': 'ACTIVETIM,AD,ADJUST,ALERT,ALGO,ALLOC,AVGCOST,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,ICE,IOC,LIT,LMT,LTH,MIT,MKT,MTL,NGCOMB,NONALGO,OCA,PEGBENCH,SCALE,SCALERST,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF',
   'validExchanges': 'CME',
   'priceMagnifier': 1,
   'underConId': 498814298,
   'longName': 'Micro Bitcoin',
   'contractMonth': '202501',
   'timeZoneId': 'US/Central',
   'tradingHours': '20250111:CLOSED;20250112:1700-20250113:1600;20250113:1700-20250114:1600;20250114:1700-20250115:1600;20250115:1700-20250116

In [20]:
from datetime import datetime
from datetime import datetime, time
import zoneinfo

def parse_hours(hours_str, timeZoneId):
    hours_list = []
    tz = zoneinfo.ZoneInfo(timeZoneId)
    for period in hours_str.split(';'):
        t = period.split(':')
        z = period.split('-')
        # print(f"t {t}, z {z}")
        if len(t) == 2 and len(z) == 1:
            if t[1] == 'CLOSED':
                date_start = date_end = t[0]
                hour_start = hour_end = None
                hours_list.append({
                    'period_str': period,
                    'date_start': date_start,
                    'start_trading_hour': None,
                    'date_end': date_end,
                    'end_trading_hour': None,
                    'is_closed': True
                })
            else:
                print("format unknown: {period}")
        elif len(t) == 3 and len(z) == 2:
            date_start, hour_start = z[0].split(':')
            date_end, hour_end = z[1].split(':')
            date_start = datetime.strptime(date_start, r'%Y%m%d').date()
            hour_start = datetime.strptime(hour_start, r'%H%M').time()
            date_end = datetime.strptime(date_end, r'%Y%m%d').date()
            hour_end = datetime.strptime(hour_end, r'%H%M').time()
            hours_list.append({
                'period_str': period,
                'date_start': date_start,
                'start_trading_hour': datetime.combine(date_start, hour_start, tz),
                'date_end': date_end,
                'end_trading_hour': datetime.combine(date_end, hour_end, tz),
                'is_closed': False
            })
        else:
            print("format unknown: {period}")
        # print(f"date_start {date_start}, date_end {date_end}, hour_start {hour_start}, hour_end {hour_end}")
    print(hours_list)

    return hours_list

# Example usage
# timeZoneId = 'America/New_York'
# trading_hours = "20231001-20231001:00:00-23:59;20231002-20231002:CLOSED;20231003-20231003:00:00-23:59"
# liquid_hours = "20231001-20231001:00:00-23:59;20231002-20231002:00:00-23:59;20231003-20231003:CLOSED"

timeZoneId = 'US/Eastern'
tradingHours = '20241130:CLOSED;20241201:CLOSED;20241202:0400-20241202:2000;20241203:0400-20241203:2000;20241204:0400-20241204:2000;20241205:0400-20241205:2000'
liquidHours = '20241130:CLOSED;20241201:CLOSED;20241202:0930-20241202:1600;20241203:0930-20241203:1600;20241204:0930-20241204:1600;20241205:0930-20241205:1600'

parsed_trading_hours = parse_hours(tradingHours, timeZoneId)
parsed_liquid_hours = parse_hours(liquidHours, timeZoneId)

# print(parsed_trading_hours)
# print(parsed_liquid_hours)

[{'period_str': '20241130:CLOSED', 'date_start': '20241130', 'start_trading_hour': None, 'date_end': '20241130', 'end_trading_hour': None, 'is_closed': True}, {'period_str': '20241201:CLOSED', 'date_start': '20241201', 'start_trading_hour': None, 'date_end': '20241201', 'end_trading_hour': None, 'is_closed': True}, {'period_str': '20241202:0400-20241202:2000', 'date_start': datetime.date(2024, 12, 2), 'start_trading_hour': datetime.datetime(2024, 12, 2, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Eastern')), 'date_end': datetime.date(2024, 12, 2), 'end_trading_hour': datetime.datetime(2024, 12, 2, 20, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Eastern')), 'is_closed': False}, {'period_str': '20241203:0400-20241203:2000', 'date_start': datetime.date(2024, 12, 3), 'start_trading_hour': datetime.datetime(2024, 12, 3, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Eastern')), 'date_end': datetime.date(2024, 12, 3), 'end_trading_hour': datetime.datetime(2024, 12, 3, 20, 0, tzinfo=zoneinfo.ZoneInfo(key='US/East

In [27]:
parse_hours(s1d[0].tradingHours, s1d[0].timeZoneId)

[{'period_str': '20250111:CLOSED', 'date_start': '20250111', 'start_trading_hour': None, 'date_end': '20250111', 'end_trading_hour': None, 'is_closed': True}, {'period_str': '20250112:1700-20250113:1600', 'date_start': datetime.date(2025, 1, 12), 'start_trading_hour': datetime.datetime(2025, 1, 12, 17, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Central')), 'date_end': datetime.date(2025, 1, 13), 'end_trading_hour': datetime.datetime(2025, 1, 13, 16, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Central')), 'is_closed': False}, {'period_str': '20250113:1700-20250114:1600', 'date_start': datetime.date(2025, 1, 13), 'start_trading_hour': datetime.datetime(2025, 1, 13, 17, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Central')), 'date_end': datetime.date(2025, 1, 14), 'end_trading_hour': datetime.datetime(2025, 1, 14, 16, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Central')), 'is_closed': False}, {'period_str': '20250114:1700-20250115:1600', 'date_start': datetime.date(2025, 1, 14), 'start_trading_hour': datetime.datetime

[{'period_str': '20250111:CLOSED',
  'date_start': '20250111',
  'start_trading_hour': None,
  'date_end': '20250111',
  'end_trading_hour': None,
  'is_closed': True},
 {'period_str': '20250112:1700-20250113:1600',
  'date_start': datetime.date(2025, 1, 12),
  'start_trading_hour': datetime.datetime(2025, 1, 12, 17, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Central')),
  'date_end': datetime.date(2025, 1, 13),
  'end_trading_hour': datetime.datetime(2025, 1, 13, 16, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Central')),
  'is_closed': False},
 {'period_str': '20250113:1700-20250114:1600',
  'date_start': datetime.date(2025, 1, 13),
  'start_trading_hour': datetime.datetime(2025, 1, 13, 17, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Central')),
  'date_end': datetime.date(2025, 1, 14),
  'end_trading_hour': datetime.datetime(2025, 1, 14, 16, 0, tzinfo=zoneinfo.ZoneInfo(key='US/Central')),
  'is_closed': False},
 {'period_str': '20250114:1700-20250115:1600',
  'date_start': datetime.date(2025, 1, 14),
  's

In [ ]:
ib_insync.ib.install_custom_repr_()

In [ ]:
ib.reqContractDetails(c3)

In [ ]:
bars = ib.reqHistoricalData(
    f5,
    endDateTime='',
    durationStr='1 D',
    barSizeSetting='1 min', # '5 secs', # always ticks every 5 secs
    whatToShow='MIDPOINT', # https://interactivebrokers.github.io/tws-api/historical_bars.html#hd_what_to_show
    useRTH=False, # start from ~4:00 AM
    formatDate=1,
    keepUpToDate=True)

In [ ]:
[ s1.__repr__(), s1.__repr__orig() ]

### Sending a request

The IB class has nearly all request methods that the IB API offers. The methods that return a result will block until finished and then return the result. Take for example reqContractDetails:

In [ ]:
contract = Stock('TSLA', 'SMART', 'USD')
ib.reqContractDetails(contract)

### Current state vs request

Doing a request involves network traffic going up and down and can take considerable time. The current state on the other hand is always immediately available. So it is preferable to use the current state methods over requests. For example, use ``ib.openOrders()`` in preference over ``ib.reqOpenOrders()``, or ``ib.positions()`` over ``ib.reqPositions()``, etc:

In [ ]:
%time l1 = ib.positions()

In [ ]:
%time l2 = ib.reqPositions()

In [ ]:
# %time l3 = ib.reqPositionsMulti()

In [ ]:
util.logToConsole(logging.DEBUG)
l = await ib.reqPositionsMultiAsync('', '')

In [ ]:
asyncio.get_running_loop()

### Logging

The following will put log messages of INFO and higher level under the current active cell:

In [ ]:
util.logToConsole()

To see all debug messages (including network traffic):

In [ ]:
import logging
util.logToConsole(logging.DEBUG)

In [ ]:
type(ib.accountValueEvent)

### Disconnecting

The following will disconnect ``ib`` and clear all its state:

In [13]:
ib.disconnect()

2025-10-23 11:03:50,804 ib_insync.client DEBUG Client::isConnected
2025-10-23 11:03:50,804 ib_insync.client DEBUG Client::connectionStats
2025-10-23 11:03:50,805 ib_insync.ib INFO Disconnecting from 127.0.0.1:7497, 119 B sent in 8 messages, 25.2 kB received in 534 messages, session time 170 s.
2025-10-23 11:03:50,805 ib_insync.client INFO Disconnecting
2025-10-23 11:03:50,806 ib_insync.client DEBUG Client::reset


2025-10-23 11:03:50,809 ib_insync.client DEBUG Client::isConnected
2025-10-23 11:03:50,809 ib_insync.client INFO Disconnected.
2025-10-23 11:03:50,810 ib_insync.client DEBUG Client::reset
